In [2]:
"""
CodeT5+ C# CQRS code generation script.

Loads a pre-trained CodeT5+ model and generates C# CQRS boilerplate files
from natural-language prompts, saving each result to the output directory.
"""

from __future__ import annotations

import os
import sys

import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration


# ──────────────────────────────────────────────────────────────
# Tokenizer compatibility patch  (transformers ≥ 4.48 Rust backend)
# ──────────────────────────────────────────────────────────────

def _patch_tokenizer_backend() -> None:
    """Fix AddedToken compatibility between the Python and Rust tokenizer backends."""
    try:
        from transformers.tokenization_utils_tokenizers import TokenizersBackend
        from tokenizers import AddedToken as RustAddedToken

        def _add_tokens_fixed(self, new_tokens: list, special_tokens: bool = False) -> int:
            converted = []
            for token in new_tokens:
                if isinstance(token, (str, RustAddedToken)):
                    converted.append(token)
                elif hasattr(token, "content"):
                    kwargs = dict(
                        single_word=getattr(token, "single_word", False),
                        lstrip=getattr(token, "lstrip", False),
                        rstrip=getattr(token, "rstrip", False),
                        normalized=getattr(token, "normalized", True),
                    )
                    try:
                        converted.append(RustAddedToken(str(token.content), special=special_tokens, **kwargs))
                    except Exception:
                        converted.append(RustAddedToken(str(token.content), **kwargs))
                else:
                    converted.append(str(token))

            return (
                self._tokenizer.add_special_tokens(converted)
                if special_tokens
                else self._tokenizer.add_tokens(converted)
            )

        TokenizersBackend._add_tokens = _add_tokens_fixed

    except ImportError:
        pass


_patch_tokenizer_backend()


# ──────────────────────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────────────────────

_MODEL_PATH:        str = os.path.abspath("../../content/codet5p-770m")
_OUTPUT_DIR:        str = os.path.abspath("../../content/generated/base")
_MAX_INPUT_TOKENS:  int = 512
_MAX_OUTPUT_TOKENS: int = 2048

# Each entry is (prompt_text, output_filename).
_TEST_PROMPTS: list[tuple[str, str]] = [
    ("CreateClaimCommand fields: Date, ReferenceId",              "CreateClaimCommand.cs"),
    ("CreateClaimCommandValidator fields: Date, ReferenceId",     "CreateClaimCommandValidator.cs"),
    ("Claim Entity fields: Id, Date, ReferenceId",                "Claim.cs"),
    ("UpdateClaimCommand fields: Id, Date, ReferenceId",          "UpdateClaimCommand.cs"),
    ("UpdateClaimCommandValidator fields: Id, Date, ReferenceId", "UpdateClaimCommandValidator.cs"),
    ("GetClaimByIdQuery fields: Id",                              "GetClaimByIdQuery.cs"),
    ("GetClaimByIdQueryValidator fields: Id",                     "GetClaimByIdQueryValidator.cs"),
    ("GetAllClaimsQuery fields: Date",                            "GetAllClaimsQuery.cs"),
    ("GetAllClaimsQueryValidator fields: Date",                   "GetAllClaimsQueryValidator.cs"),
]


# ──────────────────────────────────────────────────────────────
# Model loader
# ──────────────────────────────────────────────────────────────

def load_model(model_path: str) -> tuple:
    """Load the tokenizer and T5 model from the given directory.

    Args:
        model_path: Absolute path to the model directory.

    Returns:
        Tuple of (model, tokenizer, device_string).
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Using device: {device}")

    print("[INFO] Loading tokenizer and model ...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()

    print("[INFO] Ready.\n")
    return model, tokenizer, device


# ──────────────────────────────────────────────────────────────
# Generation
# ──────────────────────────────────────────────────────────────

def generate(model, tokenizer, device: str, prompt: str) -> str:
    """Generate C# source code from a natural-language prompt.

    Args:
        model:     Loaded T5ForConditionalGeneration model.
        tokenizer: Matching AutoTokenizer.
        device:    Target device string ("cuda" or "cpu").
        prompt:    Natural-language description of the C# artifact to generate.

    Returns:
        Decoded generated source code as a plain string.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=_MAX_INPUT_TOKENS,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=_MAX_OUTPUT_TOKENS,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# ──────────────────────────────────────────────────────────────
# Output helpers
# ──────────────────────────────────────────────────────────────

def _print_output(prompt: str, output: str) -> None:
    """Pretty-print a prompt and its generated output to stdout."""
    separator = "─" * 60
    print(f"\n{separator}")
    print(f"  Prompt : {prompt}")
    print(separator)
    print(output)
    print(f"{separator}\n")


def _save_output(output: str, file_name: str, output_dir: str) -> str:
    """Write generated source code to a .cs file.

    Args:
        output:     Generated C# source code.
        file_name:  Target filename (`.cs` extension appended if missing).
        output_dir: Directory to write into (created automatically if absent).

    Returns:
        Absolute path of the saved file.
    """
    os.makedirs(output_dir, exist_ok=True)

    if not file_name.endswith(".cs"):
        file_name += ".cs"

    file_path = os.path.join(output_dir, file_name)
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(output)

    return file_path


# ──────────────────────────────────────────────────────────────
# Batch runner
# ──────────────────────────────────────────────────────────────

def run_prompts(
    model,
    tokenizer,
    device: str,
    prompts: list[tuple[str, str]],
    output_dir: str,
) -> None:
    """Generate and save a C# file for every (prompt, filename) pair.

    Args:
        model:      Loaded T5 model.
        tokenizer:  Matching tokenizer.
        device:     Target device string.
        prompts:    List of (prompt_text, output_filename) pairs.
        output_dir: Directory where generated .cs files are saved.
    """
    total = len(prompts)

    for index, (prompt_text, file_name) in enumerate(prompts, start=1):
        print(f"[INFO] ({index}/{total}) Generating: {file_name}")

        try:
            output = generate(model, tokenizer, device, prompt_text)
        except Exception as exc:
            print(f"[ERROR] Generation failed for '{file_name}': {exc}")
            continue

        if not output.strip():
            print(f"[WARN]  Empty output for prompt: {prompt_text!r} — skipping.")
            continue

        saved_path = _save_output(output, file_name, output_dir)
        _print_output(prompt_text, output)
        print(f"[INFO] Saved → {saved_path}")

    print(f"\n[INFO] Done. {total} file(s) written to: {output_dir}")


# ──────────────────────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────────────────────

def main() -> None:
    model, tokenizer, device = load_model(_MODEL_PATH)
    run_prompts(model, tokenizer, device, _TEST_PROMPTS, _OUTPUT_DIR)


if __name__ == "__main__":
    main()

[INFO] Using device: cpu
[INFO] Loading tokenizer and model ...


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 2563.90it/s]


[INFO] Ready.

[INFO] (1/9) Generating: CreateClaimCommand.cs

────────────────────────────────────────────────────────────
  Prompt : CreateClaimCommand fields: Date, ReferenceId
────────────────────────────────────────────────────────────
, ReferenceId

type ClaimCommand struct {
	Command
}

────────────────────────────────────────────────────────────

[INFO] Saved → D:\PycharmProjects\nlp-project-codet5base-fine-tuning-cqrs-features\content\generated\base\CreateClaimCommand.cs
[INFO] (2/9) Generating: CreateClaimCommandValidator.cs

────────────────────────────────────────────────────────────
  Prompt : CreateClaimCommandValidator fields: Date, ReferenceId
────────────────────────────────────────────────────────────
, ReferenceId, ClaimCommandValidator> {

    @Override
	public void validate(ClaimCommand command, ValidationContext validationContext) throws ValidationFailedException{
    	if(command.getClaimId() == null){
        	validationContext.addError("claimId", "The claimId is